In [1]:
{
  "review": "...",
  "sentiment": "positive",
  "emotion": "joy",
  "topics": ["product quality", "delivery"],
  "aspects": [
    {
      "aspect": "quality",
      "sentiment": "Positive",
      "confidence": 0.97
    }
  ]
}

{'review': '...',
 'sentiment': 'positive',
 'emotion': 'joy',
 'topics': ['product quality', 'delivery'],
 'aspects': [{'aspect': 'quality',
   'sentiment': 'Positive',
   'confidence': 0.97}]}

In [2]:
import pandas as pd
import torch

from transformers import pipeline

print("Pipeline environment ready. ✅")
print("CUDA available:", torch.cuda.is_available())

c:\Users\sondo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pipeline environment ready. ✅
CUDA available: True


In [3]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)")

SENTIMENT_MODEL = PROJECT_ROOT / "models" / "sentiment" / "bert_binary_weighted"
EMOTION_MODEL = PROJECT_ROOT / "models" / "emotion" / "bert_emotion"
ABSA_MODEL = "yangheng/deberta-v3-base-absa-v1.1"

print("Sentiment model:", SENTIMENT_MODEL)
print("Exists:", SENTIMENT_MODEL.exists())

print("Emotion model:", EMOTION_MODEL)
print("Exists:", EMOTION_MODEL.exists())

print("ABSA model:", ABSA_MODEL)

Sentiment model: D:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\models\sentiment\bert_binary_weighted
Exists: True
Emotion model: D:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\models\emotion\bert_emotion
Exists: True
ABSA model: yangheng/deberta-v3-base-absa-v1.1


In [4]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Sentiment
sentiment_tokenizer = AutoTokenizer.from_pretrained(SENTIMENT_MODEL)
sentiment_model = AutoModelForSequenceClassification.from_pretrained(
    SENTIMENT_MODEL
).to("cuda")

# Emotion
emotion_tokenizer = AutoTokenizer.from_pretrained(EMOTION_MODEL)
emotion_model = AutoModelForSequenceClassification.from_pretrained(
    EMOTION_MODEL
).to("cuda")

sentiment_model.eval()
emotion_model.eval()

print("Sentiment model loaded. ")
print("Emotion model loaded. ")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4640.18it/s]


Sentiment model loaded. 
Emotion model loaded. 


In [5]:
print("Sentiment labels:")
print(sentiment_model.config.id2label)

print("\nEmotion labels:")
print(emotion_model.config.id2label)

Sentiment labels:
{0: 'negative', 1: 'positive'}

Emotion labels:
{0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}


In [6]:
from transformers import pipeline

absa_classifier = pipeline(
    "text-classification",
    model=ABSA_MODEL,
    tokenizer=ABSA_MODEL,
    device=0 if torch.cuda.is_available() else -1
)

print("ABSA model loaded. ✅")

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 5063.19it/s]


ABSA model loaded. ✅


In [7]:
test_review = (
    "The product is excellent and easy to use. "
    "The battery lasts a long time, but the delivery was slow."
)

# Sentiment
sentiment_output = sentiment_model(
    **sentiment_tokenizer(
        test_review,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to("cuda")
)

sentiment_label = sentiment_model.config.id2label[
    sentiment_output.logits.argmax(-1).item()
]

# Emotion
emotion_output = emotion_model(
    **emotion_tokenizer(
        test_review,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to("cuda")
)

emotion_label = emotion_model.config.id2label[
    emotion_output.logits.argmax(-1).item()
]

# ABSA
absa_quality = absa_classifier(
    test_review,
    text_pair="quality"
)[0]

absa_battery = absa_classifier(
    test_review,
    text_pair="battery"
)[0]

print("Review:")
print(test_review)

print("\nSentiment:", sentiment_label)
print("Emotion:", emotion_label)

print("\nABSA - quality:")
print(absa_quality)

print("\nABSA - battery:")
print(absa_battery)

Review:
The product is excellent and easy to use. The battery lasts a long time, but the delivery was slow.

Sentiment: negative
Emotion: fear

ABSA - quality:
{'label': 'Positive', 'score': 0.4896906912326813}

ABSA - battery:
{'label': 'Positive', 'score': 0.98460453748703}


In [8]:
def predict_sentiment(text):
    inputs = sentiment_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to("cuda")

    with torch.no_grad():
        outputs = sentiment_model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=-1)
    predicted_id = probabilities.argmax(dim=-1).item()

    return {
        "label": sentiment_model.config.id2label[predicted_id],
        "confidence": round(probabilities[0, predicted_id].item(), 4)
    }


print(predict_sentiment(test_review))

{'label': 'negative', 'confidence': 0.5006}


In [9]:
def predict_emotion(text):
    inputs = emotion_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128
    ).to("cuda")

    with torch.no_grad():
        outputs = emotion_model(**inputs)

    probabilities = torch.softmax(outputs.logits, dim=-1)
    predicted_id = probabilities.argmax(dim=-1).item()

    return {
        "label": emotion_model.config.id2label[predicted_id],
        "confidence": round(probabilities[0, predicted_id].item(), 4)
    }


print(predict_emotion(test_review))

{'label': 'fear', 'confidence': 0.7052}


In [10]:
import pandas as pd
from pathlib import Path

def load_reviews_from_excel(file_path):
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    df = pd.read_excel(file_path)

    print(f"Loaded {len(df)} reviews.")
    print("Columns:", list(df.columns))

    return df

In [11]:
sample_reviews = pd.DataFrame({
    "review_id": [1, 2, 3, 4, 5],
    "review": [
        "The product is excellent and easy to use. The battery lasts a long time.",
        "The delivery was very slow and the packaging was damaged.",
        "Great quality and beautiful design. I really love this product.",
        "The size is too small and the fit is uncomfortable.",
        "The product works well, but the price is too high."
    ]
})

sample_file = PROJECT_ROOT / "data" / "raw" / "sample_reviews.xlsx"

sample_file.parent.mkdir(parents=True, exist_ok=True)

sample_reviews.to_excel(
    sample_file,
    index=False
)

print("Sample Excel created successfully. ✅")
print("File:", sample_file)

ModuleNotFoundError: No module named 'openpyxl'

In [12]:
%pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
sample_reviews.to_excel(
    sample_file,
    index=False
)

print("Sample Excel created successfully. ✅")
print("File:", sample_file)

Sample Excel created successfully. ✅
File: D:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\data\raw\sample_reviews.xlsx


In [14]:
reviews_df = load_reviews_from_excel(sample_file)

print("\nPreview:")
display(reviews_df.head())

print("\nShape:", reviews_df.shape)

Loaded 5 reviews.
Columns: ['review_id', 'review']

Preview:


,review_id,review
0,1,The product is excellent and easy to use. The ...
1,2,The delivery was very slow and the packaging w...
2,3,Great quality and beautiful design. I really l...
3,4,The size is too small and the fit is uncomfort...
4,5,"The product works well, but the price is too h..."



Shape: (5, 2)


In [15]:
def prepare_reviews(df):
    df = df.copy()

    # التأكد من وجود عمود review
    if "review" not in df.columns:
        raise ValueError(
            "The input file must contain a 'review' column."
        )

    # الاحتفاظ بالأعمدة المطلوبة فقط
    if "review_id" not in df.columns:
        df.insert(0, "review_id", range(1, len(df) + 1))

    df = df[["review_id", "review"]]

    # تنظيف النصوص
    df["review"] = (
        df["review"]
        .astype(str)
        .str.strip()
    )

    # حذف الـreviews الفارغة
    df = df[df["review"].str.len() > 0].reset_index(drop=True)

    return df


reviews_df = prepare_reviews(reviews_df)

print("Prepared reviews:", len(reviews_df))
display(reviews_df)

Prepared reviews: 5


,review_id,review
0,1,The product is excellent and easy to use. The ...
1,2,The delivery was very slow and the packaging w...
2,3,Great quality and beautiful design. I really l...
3,4,The size is too small and the fit is uncomfort...
4,5,"The product works well, but the price is too h..."


In [16]:
reviews_df["sentiment"] = reviews_df["review"].apply(
    lambda text: predict_sentiment(text)["label"]
)

reviews_df["sentiment_confidence"] = reviews_df["review"].apply(
    lambda text: predict_sentiment(text)["confidence"]
)

display(
    reviews_df[
        ["review_id", "review", "sentiment", "sentiment_confidence"]
    ]
)

,review_id,review,sentiment,sentiment_confidence
0,1,The product is excellent and easy to use. The ...,positive,0.9586
1,2,The delivery was very slow and the packaging w...,negative,0.9909
2,3,Great quality and beautiful design. I really l...,positive,0.9877
3,4,The size is too small and the fit is uncomfort...,negative,0.9945
4,5,"The product works well, but the price is too h...",negative,0.8289


In [17]:
emotion_results = reviews_df["review"].apply(predict_emotion)

reviews_df["emotion"] = emotion_results.apply(lambda x: x["label"])
reviews_df["emotion_confidence"] = emotion_results.apply(
    lambda x: x["confidence"]
)

display(
    reviews_df[
        [
            "review_id",
            "review",
            "sentiment",
            "sentiment_confidence",
            "emotion",
            "emotion_confidence"
        ]
    ]
)

,review_id,review,sentiment,sentiment_confidence,emotion,emotion_confidence
0,1,The product is excellent and easy to use. The ...,positive,0.9586,joy,0.9991
1,2,The delivery was very slow and the packaging w...,negative,0.9909,sadness,0.8955
2,3,Great quality and beautiful design. I really l...,positive,0.9877,joy,0.9979
3,4,The size is too small and the fit is uncomfort...,negative,0.9945,fear,0.9990
4,5,"The product works well, but the price is too h...",negative,0.8289,joy,0.9922


In [18]:
from bertopic import BERTopic

BER_TOPIC_MODEL = PROJECT_ROOT / "models" / "topics" / "bertopic_final"

topic_model = BERTopic.load(
    str(BER_TOPIC_MODEL)
)

print("BERTopic model loaded. ✅")

2026-09-26 14:16:39,010 - BERTopic - WARNING: You are loading a BERTopic model without explicitly defining an embedding model. If you want to also load in an embedding model, make sure to use `BERTopic.load(my_model, embedding_model=my_embedding_model)`.


BERTopic model loaded. ✅


In [19]:
from sentence_transformers import SentenceTransformer

topic_embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cuda"
)

topic_model.embedding_model = topic_embedding_model

print("BERTopic embedding model attached. ✅")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3774.29it/s]


BERTopic embedding model attached. ✅


In [21]:
review_embeddings = topic_embedding_model.encode(
    reviews_df["review"].tolist(),
    show_progress_bar=False,
    convert_to_numpy=True
)

print("Embeddings shape:", review_embeddings.shape)

Embeddings shape: (5, 384)


In [23]:
topic_predictions, topic_probabilities = topic_model.transform(
    reviews_df["review"].tolist(),
    embeddings=review_embeddings
)

reviews_df["topic_id"] = topic_predictions

display(
    reviews_df[
        ["review_id", "review", "topic_id"]
    ]
)

2026-09-26 14:23:41,171 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


,review_id,review,topic_id
0,1,The product is excellent and easy to use. The ...,10
1,2,The delivery was very slow and the packaging w...,5
2,3,Great quality and beautiful design. I really l...,0
3,4,The size is too small and the fit is uncomfort...,2
4,5,"The product works well, but the price is too h...",1


In [24]:
topic_info = topic_model.get_topic_info()

for topic_id in reviews_df["topic_id"].unique():
    topic_row = topic_info[topic_info["Topic"] == topic_id]

    if not topic_row.empty:
        print(
            f"Topic {topic_id}: "
            f"{topic_row.iloc[0]['Name']}"
        )

Topic 10: 10_easy_camera_install_use
Topic 5: 5_received_item_box_arrived
Topic 0: 0_bag_great_quality_watch
Topic 2: 2_size_comfortable_fit_shoes
Topic 1: 1_sound_work_charge_battery


In [25]:
topic_name_map = (
    topic_info
    .set_index("Topic")["Name"]
    .to_dict()
)

reviews_df["topic"] = reviews_df["topic_id"].map(topic_name_map)

display(
    reviews_df[
        ["review_id", "review", "topic_id", "topic"]
    ]
)

,review_id,review,topic_id,topic
0,1,The product is excellent and easy to use. The ...,10,10_easy_camera_install_use
1,2,The delivery was very slow and the packaging w...,5,5_received_item_box_arrived
2,3,Great quality and beautiful design. I really l...,0,0_bag_great_quality_watch
3,4,The size is too small and the fit is uncomfort...,2,2_size_comfortable_fit_shoes
4,5,"The product works well, but the price is too h...",1,1_sound_work_charge_battery


In [27]:
def extract_review_aspects(text):
    return extract_aspects(text, ASPECT_TERMS)

print("Aspect extraction helper restored. ✅")

Aspect extraction helper restored. ✅


In [31]:
reviews_df["aspects"] = reviews_df["review"].apply(
    extract_review_aspects
)

display(
    reviews_df[
        ["review_id", "review", "aspects"]
    ]
)

,review_id,review,aspects
0,1,The product is excellent and easy to use. The ...,"[battery, ease_of_use]"
1,2,The delivery was very slow and the packaging w...,"[delivery, packaging]"
2,3,Great quality and beautiful design. I really l...,"[quality, design]"
3,4,The size is too small and the fit is uncomfort...,"[size, fit]"
4,5,"The product works well, but the price is too h...","[price, performance]"


In [30]:
import re

ASPECT_TERMS = {
    "delivery": ["delivery", "shipping", "shipment", "arrived", "arrival", "delivered", "deliver"],
    "quality": ["quality", "durable", "durability", "material"],
    "price": ["price", "cost", "value", "expensive", "cheap"],
    "size": ["size", "sizing", "small", "large", "bigger", "smaller", "length", "width"],
    "fit": ["fit", "fits", "fitting", "comfortable", "comfort"],
    "battery": ["battery", "charging", "charge", "charger"],
    "sound": ["sound", "audio", "volume", "bass", "noise"],
    "customer_service": ["service", "support", "customer service"],
    "packaging": ["packaging", "package", "box"],
    "design": ["design", "style", "appearance", "look"],
    "performance": ["performance", "works", "working", "function", "speed"],
    "ease_of_use": ["easy", "easier", "simple", "use", "install", "installation"]
}


def extract_review_aspects(text):
    text = str(text).lower()
    detected = []

    for aspect, terms in ASPECT_TERMS.items():
        for term in terms:
            if re.search(rf"\b{re.escape(term.lower())}\b", text):
                detected.append(aspect)
                break

    return detected


print("Aspect extraction pipeline restored. ✅")

Aspect extraction pipeline restored. ✅


In [32]:
def predict_aspect_sentiments(text, aspects):
    results = []

    for aspect in aspects:
        result = absa_classifier(
            text,
            text_pair=aspect
        )[0]

        confidence = result["score"]

        if confidence >= 0.90:
            confidence_level = "high"
        elif confidence >= 0.70:
            confidence_level = "medium"
        else:
            confidence_level = "low"

        results.append({
            "aspect": aspect,
            "sentiment": result["label"],
            "confidence": round(confidence, 4),
            "confidence_level": confidence_level
        })

    return results


reviews_df["aspect_sentiments"] = reviews_df.apply(
    lambda row: predict_aspect_sentiments(
        row["review"],
        row["aspects"]
    ),
    axis=1
)

display(
    reviews_df[
        ["review_id", "review", "aspects", "aspect_sentiments"]
    ]
)

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,review_id,review,aspects,aspect_sentiments
0,1,The product is excellent and easy to use. The ...,"[battery, ease_of_use]","[{'aspect': 'battery', 'sentiment': 'Positive'..."
1,2,The delivery was very slow and the packaging w...,"[delivery, packaging]","[{'aspect': 'delivery', 'sentiment': 'Negative..."
2,3,Great quality and beautiful design. I really l...,"[quality, design]","[{'aspect': 'quality', 'sentiment': 'Positive'..."
3,4,The size is too small and the fit is uncomfort...,"[size, fit]","[{'aspect': 'size', 'sentiment': 'Negative', '..."
4,5,"The product works well, but the price is too h...","[price, performance]","[{'aspect': 'price', 'sentiment': 'Negative', ..."


In [33]:
final_results = reviews_df[
    [
        "review_id",
        "review",
        "sentiment",
        "sentiment_confidence",
        "emotion",
        "emotion_confidence",
        "topic",
        "aspect_sentiments"
    ]
].copy()

display(final_results)

,review_id,review,sentiment,sentiment_confidence,emotion,emotion_confidence,topic,aspect_sentiments
0,1,The product is excellent and easy to use. The ...,positive,0.9586,joy,0.9991,10_easy_camera_install_use,"[{'aspect': 'battery', 'sentiment': 'Positive'..."
1,2,The delivery was very slow and the packaging w...,negative,0.9909,sadness,0.8955,5_received_item_box_arrived,"[{'aspect': 'delivery', 'sentiment': 'Negative..."
2,3,Great quality and beautiful design. I really l...,positive,0.9877,joy,0.9979,0_bag_great_quality_watch,"[{'aspect': 'quality', 'sentiment': 'Positive'..."
3,4,The size is too small and the fit is uncomfort...,negative,0.9945,fear,0.9990,2_size_comfortable_fit_shoes,"[{'aspect': 'size', 'sentiment': 'Negative', '..."
4,5,"The product works well, but the price is too h...",negative,0.8289,joy,0.9922,1_sound_work_charge_battery,"[{'aspect': 'price', 'sentiment': 'Negative', ..."


In [34]:
import json

final_results["aspect_sentiments"] = final_results["aspect_sentiments"].apply(
    json.dumps
)

display(final_results)

,review_id,review,sentiment,sentiment_confidence,emotion,emotion_confidence,topic,aspect_sentiments
0,1,The product is excellent and easy to use. The ...,positive,0.9586,joy,0.9991,10_easy_camera_install_use,"[{""aspect"": ""battery"", ""sentiment"": ""Positive""..."
1,2,The delivery was very slow and the packaging w...,negative,0.9909,sadness,0.8955,5_received_item_box_arrived,"[{""aspect"": ""delivery"", ""sentiment"": ""Negative..."
2,3,Great quality and beautiful design. I really l...,positive,0.9877,joy,0.9979,0_bag_great_quality_watch,"[{""aspect"": ""quality"", ""sentiment"": ""Positive""..."
3,4,The size is too small and the fit is uncomfort...,negative,0.9945,fear,0.9990,2_size_comfortable_fit_shoes,"[{""aspect"": ""size"", ""sentiment"": ""Negative"", ""..."
4,5,"The product works well, but the price is too h...",negative,0.8289,joy,0.9922,1_sound_work_charge_battery,"[{""aspect"": ""price"", ""sentiment"": ""Negative"", ..."


In [35]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

output_file = OUTPUT_DIR / "customer_feedback_analysis.xlsx"

final_results.to_excel(
    output_file,
    index=False
)

print("Analysis file created successfully. ✅")
print("Saved to:", output_file)

Analysis file created successfully. ✅
Saved to: D:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\data\processed\customer_feedback_analysis.xlsx


In [36]:
sentiment_summary = (
    final_results["sentiment"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="count")
)

sentiment_summary["percentage"] = (
    sentiment_summary["count"]
    / len(final_results)
    * 100
).round(2)

display(sentiment_summary)

,sentiment,count,percentage
0,negative,3,60.0
1,positive,2,40.0


In [37]:
emotion_summary = (
    final_results["emotion"]
    .value_counts()
    .rename_axis("emotion")
    .reset_index(name="count")
)

emotion_summary["percentage"] = (
    emotion_summary["count"]
    / len(final_results)
    * 100
).round(2)

display(emotion_summary)

,emotion,count,percentage
0,joy,3,60.0
1,sadness,1,20.0
2,fear,1,20.0


In [38]:
topic_summary = (
    final_results["topic"]
    .value_counts()
    .rename_axis("topic")
    .reset_index(name="count")
)

topic_summary["percentage"] = (
    topic_summary["count"]
    / len(final_results)
    * 100
).round(2)

display(topic_summary)

,topic,count,percentage
0,10_easy_camera_install_use,1,20.0
1,5_received_item_box_arrived,1,20.0
2,0_bag_great_quality_watch,1,20.0
3,2_size_comfortable_fit_shoes,1,20.0
4,1_sound_work_charge_battery,1,20.0


In [39]:
aspect_rows = []

for _, row in final_results.iterrows():
    aspect_data = json.loads(row["aspect_sentiments"])

    for item in aspect_data:
        aspect_rows.append({
            "review_id": row["review_id"],
            "aspect": item["aspect"],
            "sentiment": item["sentiment"].lower(),
            "confidence": item["confidence"],
            "confidence_level": item["confidence_level"]
        })

aspect_results = pd.DataFrame(aspect_rows)

display(aspect_results)

,review_id,aspect,sentiment,confidence,confidence_level
0,1,battery,positive,0.9921,high
1,1,ease_of_use,positive,0.9970,high
2,2,delivery,negative,0.9977,high
3,2,packaging,negative,0.9974,high
4,3,quality,positive,0.9980,high
5,3,design,positive,0.9982,high
6,4,size,negative,0.9960,high
7,4,fit,negative,0.9957,high
8,5,price,negative,0.9947,high
9,5,performance,positive,0.9868,high


In [40]:
aspect_summary = (
    aspect_results
    .groupby(["aspect", "sentiment"])
    .agg(
        count=("review_id", "count"),
        avg_confidence=("confidence", "mean")
    )
    .reset_index()
)

aspect_summary["avg_confidence"] = (
    aspect_summary["avg_confidence"]
    .round(4)
)

display(
    aspect_summary.sort_values(
        ["aspect", "count"],
        ascending=[True, False]
    )
)

,aspect,sentiment,count,avg_confidence
0,battery,positive,1,0.9921
1,delivery,negative,1,0.9977
2,design,positive,1,0.9982
3,ease_of_use,positive,1,0.9970
4,fit,negative,1,0.9957
5,packaging,negative,1,0.9974
6,performance,positive,1,0.9868
7,price,negative,1,0.9947
8,quality,positive,1,0.9980
9,size,negative,1,0.9960


In [41]:
sentiment_score_map = {
    "positive": 1,
    "neutral": 0,
    "negative": -1
}

aspect_health = aspect_results.copy()

aspect_health["sentiment_score"] = (
    aspect_health["sentiment"]
    .map(sentiment_score_map)
)

aspect_health_summary = (
    aspect_health
    .groupby("aspect")
    .agg(
        mentions=("review_id", "count"),
        sentiment_score=("sentiment_score", "mean"),
        avg_confidence=("confidence", "mean")
    )
    .reset_index()
)

aspect_health_summary["sentiment_score"] = (
    aspect_health_summary["sentiment_score"]
    .round(2)
)

aspect_health_summary["avg_confidence"] = (
    aspect_health_summary["avg_confidence"]
    .round(4)
)

display(aspect_health_summary)

,aspect,mentions,sentiment_score,avg_confidence
0,battery,1,1.0,0.9921
1,delivery,1,-1.0,0.9977
2,design,1,1.0,0.9982
3,ease_of_use,1,1.0,0.9970
4,fit,1,-1.0,0.9957
5,packaging,1,-1.0,0.9974
6,performance,1,1.0,0.9868
7,price,1,-1.0,0.9947
8,quality,1,1.0,0.9980
9,size,1,-1.0,0.9960


In [42]:
aspect_sentiment_distribution = (
    aspect_results["sentiment"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="count")
)

aspect_sentiment_distribution["percentage"] = (
    aspect_sentiment_distribution["count"]
    / len(aspect_results)
    * 100
).round(2)

display(aspect_sentiment_distribution)

,sentiment,count,percentage
0,positive,5,50.0
1,negative,5,50.0


In [43]:
business_insights = {
    "total_reviews": len(final_results),

    "sentiment": {
        "positive": int(
            (final_results["sentiment"] == "positive").sum()
        ),
        "negative": int(
            (final_results["sentiment"] == "negative").sum()
        ),
        "positive_percentage": round(
            (final_results["sentiment"] == "positive").mean() * 100,
            2
        ),
        "negative_percentage": round(
            (final_results["sentiment"] == "negative").mean() * 100,
            2
        )
    },

    "emotions": emotion_summary.to_dict(orient="records"),

    "topics": topic_summary.to_dict(orient="records"),

    "aspect_sentiment": aspect_sentiment_distribution.to_dict(
        orient="records"
    )
}

business_insights

{'total_reviews': 5,
 'sentiment': {'positive': 2,
  'negative': 3,
  'positive_percentage': np.float64(40.0),
  'negative_percentage': np.float64(60.0)},
 'emotions': [{'emotion': 'joy', 'count': 3, 'percentage': 60.0},
  {'emotion': 'sadness', 'count': 1, 'percentage': 20.0},
  {'emotion': 'fear', 'count': 1, 'percentage': 20.0}],
 'topics': [{'topic': '10_easy_camera_install_use',
   'count': 1,
   'percentage': 20.0},
  {'topic': '5_received_item_box_arrived', 'count': 1, 'percentage': 20.0},
  {'topic': '0_bag_great_quality_watch', 'count': 1, 'percentage': 20.0},
  {'topic': '2_size_comfortable_fit_shoes', 'count': 1, 'percentage': 20.0},
  {'topic': '1_sound_work_charge_battery', 'count': 1, 'percentage': 20.0}],
 'aspect_sentiment': [{'sentiment': 'positive',
   'count': 5,
   'percentage': 50.0},
  {'sentiment': 'negative', 'count': 5, 'percentage': 50.0}]}

In [44]:
import json

business_insights_json = json.dumps(
    business_insights,
    indent=4,
    ensure_ascii=False,
    default=lambda x: float(x) if hasattr(x, "item") else x
)

print(business_insights_json)

{
    "total_reviews": 5,
    "sentiment": {
        "positive": 2,
        "negative": 3,
        "positive_percentage": 40.0,
        "negative_percentage": 60.0
    },
    "emotions": [
        {
            "emotion": "joy",
            "count": 3,
            "percentage": 60.0
        },
        {
            "emotion": "sadness",
            "count": 1,
            "percentage": 20.0
        },
        {
            "emotion": "fear",
            "count": 1,
            "percentage": 20.0
        }
    ],
    "topics": [
        {
            "topic": "10_easy_camera_install_use",
            "count": 1,
            "percentage": 20.0
        },
        {
            "topic": "5_received_item_box_arrived",
            "count": 1,
            "percentage": 20.0
        },
        {
            "topic": "0_bag_great_quality_watch",
            "count": 1,
            "percentage": 20.0
        },
        {
            "topic": "2_size_comfortable_fit_shoes",
            "count": 1

In [45]:
json_output_file = OUTPUT_DIR / "business_insights.json"

with open(json_output_file, "w", encoding="utf-8") as f:
    f.write(business_insights_json)

print("Business insights report created successfully. ✅")
print("Saved to:", json_output_file)

Business insights report created successfully. ✅
Saved to: D:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\data\processed\business_insights.json


In [46]:
def load_reviews_from_excel(file_path):
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"File not found: {file_path}"
        )

    df = pd.read_excel(file_path)

    column_mapping = {
        "review": "review",
        "review_body": "review",
        "text": "review"
    }

    normalized_columns = {}

    for column in df.columns:
        column_key = str(column).strip().lower()

        if column_key in column_mapping:
            normalized_columns[column] = column_mapping[column_key]

    df = df.rename(columns=normalized_columns)

    if "review" not in df.columns:
        raise ValueError(
            "Excel file must contain a review column. "
            "Supported names: review, review_body, text."
        )

    print(f"Loaded {len(df)} reviews. ✅")
    print("Columns:", list(df.columns))

    return df

In [47]:
test_excel_df = load_reviews_from_excel(sample_file)

test_excel_df = prepare_reviews(test_excel_df)

display(test_excel_df)

Loaded 5 reviews. ✅
Columns: ['review_id', 'review']


,review_id,review
0,1,The product is excellent and easy to use. The ...
1,2,The delivery was very slow and the packaging w...
2,3,Great quality and beautiful design. I really l...
3,4,The size is too small and the fit is uncomfort...
4,5,"The product works well, but the price is too h..."


In [48]:
%pip install pymupdf

   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/19.8 MB ? eta -:--:--
    --------------------------------------- 0.3/19.8 MB ? eta -:--:--
   - -------------------------------------- 0.5/19.8 MB 1.2 MB/s eta 0:00:17
   - -------------------------------------- 0.8/19.8 MB 1.3 MB/s eta 0:00:15
   -- ------------------------------------- 1.3/19.8 MB 1.4 MB/s eta 0:00:13
   --- ------------------------------------ 1.6/19.8 MB 1.4 MB/s eta 0:00:13
   --- ------------------------------------ 1.8/19.8 MB 1.5 MB/s eta 0:00:13
   ---- ----------------------------------- 2.4/19.8 MB 1.5 MB/s eta 0:00:12
   ----- ---------------------------------- 2.6/19.8 MB 1.5 MB/s eta 0:00:12
   ----- ---------------------------------- 2.9/19.8 MB 1.6 MB/s eta 0:00:11
   ------ --------------------------------- 3.4/19.8 MB 1.6 MB/s eta 0:00:11
   ------- ----------------


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [49]:
import fitz

def load_reviews_from_pdf(file_path):
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"File not found: {file_path}"
        )

    document = fitz.open(file_path)

    pages_text = []

    for page in document:
        text = page.get_text("text").strip()

        if text:
            pages_text.append(text)

    document.close()

    full_text = "\n".join(pages_text)

    if not full_text.strip():
        raise ValueError(
            "No text could be extracted from the PDF."
        )

    print("PDF loaded successfully. ✅")
    print("Extracted characters:", len(full_text))

    return full_text

In [50]:
import pymupdf

def load_reviews_from_pdf(file_path):
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"File not found: {file_path}"
        )

    document = pymupdf.open(file_path)

    pages_text = []

    for page in document:
        text = page.get_text("text").strip()

        if text:
            pages_text.append(text)

    document.close()

    full_text = "\n".join(pages_text)

    if not full_text.strip():
        raise ValueError(
            "No text could be extracted from the PDF."
        )

    print("PDF loaded successfully. ✅")
    print("Extracted characters:", len(full_text))

    return full_text

In [51]:
test_pdf = PROJECT_ROOT / "data" / "raw" / "sample_reviews.pdf"

pdf_text = load_reviews_from_pdf(test_pdf)

print(pdf_text[:1000])

FileNotFoundError: File not found: D:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\data\raw\sample_reviews.pdf

In [53]:
%pip install reportlab

   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 409.0 kB/s eta 0:00:04
   ---------- ----------------------------- 0.5/2.0 MB 409.0 kB/s eta 0:00:04
   ---------- ----------------------------- 0.5/2.0 MB 409.0 kB/s eta 0:00:04
   ---------------- ----------------------- 0.8/2.0 MB 394.8 kB/s eta 0:00:03
   ---------------- ----------------------- 0.8/2.0 MB 394.8 kB/s eta 0:00:03
   ---------------- ----------------------- 0.8/2.0 MB 394


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [54]:
from reportlab.pdfgen import canvas

sample_pdf = PROJECT_ROOT / "data" / "raw" / "sample_reviews.pdf"

c = canvas.Canvas(str(sample_pdf))

reviews = [
    "The product is excellent and easy to use. The battery lasts a long time.",
    "The delivery was very slow and the packaging was damaged.",
    "Great quality and beautiful design. I really love this product.",
    "The size is too small and the fit is uncomfortable.",
    "The product works well, but the price is too high."
]

y = 800

for i, review in enumerate(reviews, start=1):
    c.drawString(50, y, f"Review {i}:")
    y -= 20
    c.drawString(70, y, review)
    y -= 40

c.save()

print("Sample PDF created successfully. ✅")
print(sample_pdf)

Sample PDF created successfully. ✅
D:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\data\raw\sample_reviews.pdf


In [55]:
test_pdf = PROJECT_ROOT / "data" / "raw" / "sample_reviews.pdf"

pdf_text = load_reviews_from_pdf(test_pdf)

print("\nExtracted text:")
print(pdf_text)

PDF loaded successfully. ✅
Extracted characters: 347

Extracted text:
Review 1:
The product is excellent and easy to use. The battery lasts a long time.
Review 2:
The delivery was very slow and the packaging was damaged.
Review 3:
Great quality and beautiful design. I really love this product.
Review 4:
The size is too small and the fit is uncomfortable.
Review 5:
The product works well, but the price is too high.


In [56]:
import re
import pandas as pd

def parse_pdf_reviews(pdf_text):
    pattern = r"Review\s+(\d+):\s*(.*?)(?=Review\s+\d+:|$)"

    matches = re.findall(
        pattern,
        pdf_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if not matches:
        raise ValueError(
            "Could not detect reviews in the PDF. "
            "Expected format like: Review 1: ..."
        )

    reviews = pd.DataFrame(
        matches,
        columns=["review_id", "review"]
    )

    reviews["review_id"] = reviews["review_id"].astype(int)
    reviews["review"] = (
        reviews["review"]
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    reviews = reviews[
        reviews["review"].str.len() > 0
    ].reset_index(drop=True)

    print(f"Parsed {len(reviews)} reviews successfully. ✅")

    return reviews

In [57]:
parsed_reviews = parse_pdf_reviews(pdf_text)

display(parsed_reviews)

Parsed 5 reviews successfully. ✅


,review_id,review
0,1,The product is excellent and easy to use. The ...
1,2,The delivery was very slow and the packaging w...
2,3,Great quality and beautiful design. I really l...
3,4,The size is too small and the fit is uncomfort...
4,5,"The product works well, but the price is too h..."


In [58]:
pdf_reviews_df = prepare_reviews(parsed_reviews)

display(pdf_reviews_df)

,review_id,review
0,1,The product is excellent and easy to use. The ...
1,2,The delivery was very slow and the packaging w...
2,3,Great quality and beautiful design. I really l...
3,4,The size is too small and the fit is uncomfort...
4,5,"The product works well, but the price is too h..."


In [59]:
def load_reviews(file_path):
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(
            f"File not found: {file_path}"
        )

    suffix = file_path.suffix.lower()

    if suffix in [".xlsx", ".xls"]:
        df = load_reviews_from_excel(file_path)
        df = prepare_reviews(df)

    elif suffix == ".pdf":
        pdf_text = load_reviews_from_pdf(file_path)
        df = parse_pdf_reviews(pdf_text)
        df = prepare_reviews(df)

    else:
        raise ValueError(
            "Unsupported file format. "
            "Supported formats: .xlsx, .xls, .pdf"
        )

    print(f"Loaded {len(df)} reviews successfully. ✅")

    return df

In [60]:
pdf_test_df = load_reviews(
    PROJECT_ROOT / "data" / "raw" / "sample_reviews.pdf"
)

display(pdf_test_df)

PDF loaded successfully. ✅
Extracted characters: 347
Parsed 5 reviews successfully. ✅
Loaded 5 reviews successfully. ✅


,review_id,review
0,1,The product is excellent and easy to use. The ...
1,2,The delivery was very slow and the packaging w...
2,3,Great quality and beautiful design. I really l...
3,4,The size is too small and the fit is uncomfort...
4,5,"The product works well, but the price is too h..."


In [61]:
excel_test_df = load_reviews(
    PROJECT_ROOT / "data" / "raw" / "sample_reviews.xlsx"
)

display(excel_test_df)

Loaded 5 reviews. ✅
Columns: ['review_id', 'review']
Loaded 5 reviews successfully. ✅


,review_id,review
0,1,The product is excellent and easy to use. The ...
1,2,The delivery was very slow and the packaging w...
2,3,Great quality and beautiful design. I really l...
3,4,The size is too small and the fit is uncomfort...
4,5,"The product works well, but the price is too h..."


In [62]:
def run_text_analysis(df):
    results = df.copy()

    results["sentiment_result"] = results["review"].apply(
        predict_sentiment
    )

    results["sentiment"] = results["sentiment_result"].apply(
        lambda x: x["label"]
    )

    results["sentiment_confidence"] = results["sentiment_result"].apply(
        lambda x: x["confidence"]
    )

    results["emotion_result"] = results["review"].apply(
        predict_emotion
    )

    results["emotion"] = results["emotion_result"].apply(
        lambda x: x["label"]
    )

    results["emotion_confidence"] = results["emotion_result"].apply(
        lambda x: x["confidence"]
    )

    results = results.drop(
        columns=["sentiment_result", "emotion_result"]
    )

    print(f"Text analysis completed for {len(results)} reviews. ✅")

    return results

In [63]:
pdf_text_analysis = run_text_analysis(pdf_test_df)

display(
    pdf_text_analysis[
        [
            "review_id",
            "review",
            "sentiment",
            "sentiment_confidence",
            "emotion",
            "emotion_confidence"
        ]
    ]
)

Text analysis completed for 5 reviews. ✅


,review_id,review,sentiment,sentiment_confidence,emotion,emotion_confidence
0,1,The product is excellent and easy to use. The ...,positive,0.9586,joy,0.9991
1,2,The delivery was very slow and the packaging w...,negative,0.9909,sadness,0.8955
2,3,Great quality and beautiful design. I really l...,positive,0.9877,joy,0.9979
3,4,The size is too small and the fit is uncomfort...,negative,0.9945,fear,0.9990
4,5,"The product works well, but the price is too h...",negative,0.8289,joy,0.9922


In [64]:
def run_topic_analysis(df):
    results = df.copy()

    texts = results["review"].tolist()

    embeddings = topic_embedding_model.encode(
        texts,
        show_progress_bar=False,
        convert_to_numpy=True
    )

    topic_ids, _ = topic_model.transform(
        texts,
        embeddings=embeddings
    )

    results["topic_id"] = topic_ids

    results["topic"] = results["topic_id"].map(
        topic_name_map
    )

    print(f"Topic analysis completed for {len(results)} reviews. ✅")

    return results

In [65]:
pdf_topic_analysis = run_topic_analysis(pdf_text_analysis)

display(
    pdf_topic_analysis[
        [
            "review_id",
            "review",
            "topic_id",
            "topic"
        ]
    ]
)


2026-09-26 15:14:33,781 - BERTopic - Predicting topic assignments through cosine similarity of topic and document embeddings.


Topic analysis completed for 5 reviews. ✅


,review_id,review,topic_id,topic
0,1,The product is excellent and easy to use. The ...,10,10_easy_camera_install_use
1,2,The delivery was very slow and the packaging w...,5,5_received_item_box_arrived
2,3,Great quality and beautiful design. I really l...,0,0_bag_great_quality_watch
3,4,The size is too small and the fit is uncomfort...,2,2_size_comfortable_fit_shoes
4,5,"The product works well, but the price is too h...",1,1_sound_work_charge_battery


In [66]:
def run_aspect_analysis(df):
    results = df.copy()

    results["aspects"] = results["review"].apply(
        extract_review_aspects
    )

    results["aspect_sentiments"] = results.apply(
        lambda row: predict_aspect_sentiments(
            row["review"],
            row["aspects"]
        ),
        axis=1
    )

    print(
        f"Aspect analysis completed for {len(results)} reviews. ✅"
    )

    return results

In [67]:
pdf_aspect_analysis = run_aspect_analysis(
    pdf_topic_analysis
)

display(
    pdf_aspect_analysis[
        [
            "review_id",
            "review",
            "aspects",
            "aspect_sentiments"
        ]
    ]
)

Aspect analysis completed for 5 reviews. ✅


,review_id,review,aspects,aspect_sentiments
0,1,The product is excellent and easy to use. The ...,"[battery, ease_of_use]","[{'aspect': 'battery', 'sentiment': 'Positive'..."
1,2,The delivery was very slow and the packaging w...,"[delivery, packaging]","[{'aspect': 'delivery', 'sentiment': 'Negative..."
2,3,Great quality and beautiful design. I really l...,"[quality, design]","[{'aspect': 'quality', 'sentiment': 'Positive'..."
3,4,The size is too small and the fit is uncomfort...,"[size, fit]","[{'aspect': 'size', 'sentiment': 'Negative', '..."
4,5,"The product works well, but the price is too h...","[price, performance]","[{'aspect': 'price', 'sentiment': 'Negative', ..."


In [68]:
final_pipeline_results = pdf_aspect_analysis[
    [
        "review_id",
        "review",
        "sentiment",
        "sentiment_confidence",
        "emotion",
        "emotion_confidence",
        "topic_id",
        "topic",
        "aspects",
        "aspect_sentiments"
    ]
].copy()

display(final_pipeline_results)

,review_id,review,sentiment,sentiment_confidence,emotion,emotion_confidence,topic_id,topic,aspects,aspect_sentiments
0,1,The product is excellent and easy to use. The ...,positive,0.9586,joy,0.9991,10,10_easy_camera_install_use,"[battery, ease_of_use]","[{'aspect': 'battery', 'sentiment': 'Positive'..."
1,2,The delivery was very slow and the packaging w...,negative,0.9909,sadness,0.8955,5,5_received_item_box_arrived,"[delivery, packaging]","[{'aspect': 'delivery', 'sentiment': 'Negative..."
2,3,Great quality and beautiful design. I really l...,positive,0.9877,joy,0.9979,0,0_bag_great_quality_watch,"[quality, design]","[{'aspect': 'quality', 'sentiment': 'Positive'..."
3,4,The size is too small and the fit is uncomfort...,negative,0.9945,fear,0.9990,2,2_size_comfortable_fit_shoes,"[size, fit]","[{'aspect': 'size', 'sentiment': 'Negative', '..."
4,5,"The product works well, but the price is too h...",negative,0.8289,joy,0.9922,1,1_sound_work_charge_battery,"[price, performance]","[{'aspect': 'price', 'sentiment': 'Negative', ..."


In [69]:
import json

final_pipeline_results["aspect_sentiments"] = (
    final_pipeline_results["aspect_sentiments"]
    .apply(
        lambda x: json.dumps(
            x,
            ensure_ascii=False
        )
    )
)

display(final_pipeline_results)

,review_id,review,sentiment,sentiment_confidence,emotion,emotion_confidence,topic_id,topic,aspects,aspect_sentiments
0,1,The product is excellent and easy to use. The ...,positive,0.9586,joy,0.9991,10,10_easy_camera_install_use,"[battery, ease_of_use]","[{""aspect"": ""battery"", ""sentiment"": ""Positive""..."
1,2,The delivery was very slow and the packaging w...,negative,0.9909,sadness,0.8955,5,5_received_item_box_arrived,"[delivery, packaging]","[{""aspect"": ""delivery"", ""sentiment"": ""Negative..."
2,3,Great quality and beautiful design. I really l...,positive,0.9877,joy,0.9979,0,0_bag_great_quality_watch,"[quality, design]","[{""aspect"": ""quality"", ""sentiment"": ""Positive""..."
3,4,The size is too small and the fit is uncomfort...,negative,0.9945,fear,0.9990,2,2_size_comfortable_fit_shoes,"[size, fit]","[{""aspect"": ""size"", ""sentiment"": ""Negative"", ""..."
4,5,"The product works well, but the price is too h...",negative,0.8289,joy,0.9922,1,1_sound_work_charge_battery,"[price, performance]","[{""aspect"": ""price"", ""sentiment"": ""Negative"", ..."


In [70]:
sentiment_summary = (
    final_pipeline_results["sentiment"]
    .value_counts()
    .rename_axis("sentiment")
    .reset_index(name="count")
)

sentiment_summary["percentage"] = (
    sentiment_summary["count"]
    / len(final_pipeline_results)
    * 100
).round(2)

display(sentiment_summary)

,sentiment,count,percentage
0,negative,3,60.0
1,positive,2,40.0


In [71]:
emotion_summary = (
    final_pipeline_results["emotion"]
    .value_counts()
    .rename_axis("emotion")
    .reset_index(name="count")
)

emotion_summary["percentage"] = (
    emotion_summary["count"]
    / len(final_pipeline_results)
    * 100
).round(2)

display(emotion_summary)

,emotion,count,percentage
0,joy,3,60.0
1,sadness,1,20.0
2,fear,1,20.0


In [72]:
topic_summary = (
    final_pipeline_results["topic"]
    .value_counts()
    .rename_axis("topic")
    .reset_index(name="count")
)

topic_summary["percentage"] = (
    topic_summary["count"]
    / len(final_pipeline_results)
    * 100
).round(2)

display(topic_summary)

,topic,count,percentage
0,10_easy_camera_install_use,1,20.0
1,5_received_item_box_arrived,1,20.0
2,0_bag_great_quality_watch,1,20.0
3,2_size_comfortable_fit_shoes,1,20.0
4,1_sound_work_charge_battery,1,20.0


In [75]:
aspect_rows = []

for _, row in final_pipeline_results.iterrows():
    for aspect_result in row["aspect_sentiments"]:
        aspect_rows.append({
            "aspect": aspect_result["aspect"],
            "sentiment": aspect_result["sentiment"],
            "confidence": aspect_result["confidence"]
        })

aspect_df = pd.DataFrame(aspect_rows)

aspect_summary = (
    aspect_df["aspect"]
    .value_counts()
    .rename_axis("aspect")
    .reset_index(name="count")
)

aspect_summary["percentage"] = (
    aspect_summary["count"]
    / len(aspect_df)
    * 100
).round(2)

display(aspect_summary)

TypeError: string indices must be integers, not 'str'

In [76]:
import json

aspect_rows = []

for _, row in final_pipeline_results.iterrows():
    aspect_sentiments = json.loads(row["aspect_sentiments"])

    for aspect_result in aspect_sentiments:
        aspect_rows.append({
            "aspect": aspect_result["aspect"],
            "sentiment": aspect_result["sentiment"],
            "confidence": aspect_result["confidence"]
        })

aspect_df = pd.DataFrame(aspect_rows)

aspect_summary = (
    aspect_df["aspect"]
    .value_counts()
    .rename_axis("aspect")
    .reset_index(name="count")
)

aspect_summary["percentage"] = (
    aspect_summary["count"]
    / len(aspect_df)
    * 100
).round(2)

display(aspect_summary)

,aspect,count,percentage
0,battery,1,10.0
1,ease_of_use,1,10.0
2,delivery,1,10.0
3,packaging,1,10.0
4,quality,1,10.0
5,design,1,10.0
6,size,1,10.0
7,fit,1,10.0
8,price,1,10.0
9,performance,1,10.0


In [77]:

aspect_sentiment_summary = (
    aspect_df
    .groupby(["aspect", "sentiment"])
    .size()
    .reset_index(name="count")
    .sort_values(["aspect", "count"], ascending=[True, False])
)

display(aspect_sentiment_summary)

,aspect,sentiment,count
0,battery,Positive,1
1,delivery,Negative,1
2,design,Positive,1
3,ease_of_use,Positive,1
4,fit,Negative,1
5,packaging,Negative,1
6,performance,Positive,1
7,price,Negative,1
8,quality,Positive,1
9,size,Negative,1


In [78]:
aspect_sentiment_pivot = (
    aspect_df
    .groupby(["aspect", "sentiment"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

display(aspect_sentiment_pivot)

sentiment,aspect,Negative,Positive
0,battery,0,1
1,delivery,1,0
2,design,0,1
3,ease_of_use,0,1
4,fit,1,0
5,packaging,1,0
6,performance,0,1
7,price,1,0
8,quality,0,1
9,size,1,0


In [79]:
business_insights = []

# Overall sentiment
negative_pct = (
    (final_pipeline_results["sentiment"] == "negative").mean() * 100
)

positive_pct = (
    (final_pipeline_results["sentiment"] == "positive").mean() * 100
)

business_insights.append(
    f"Overall sentiment: {positive_pct:.1f}% positive and "
    f"{negative_pct:.1f}% negative reviews."
)

# Positive aspects
positive_aspects = aspect_df.loc[
    aspect_df["sentiment"].str.lower() == "positive",
    "aspect"
].value_counts()

for aspect, count in positive_aspects.items():
    business_insights.append(
        f"Positive feedback was detected for {aspect} "
        f"({count} mention{'s' if count != 1 else ''})."
    )

# Negative aspects
negative_aspects = aspect_df.loc[
    aspect_df["sentiment"].str.lower() == "negative",
    "aspect"
].value_counts()

for aspect, count in negative_aspects.items():
    business_insights.append(
        f"Negative feedback was detected for {aspect} "
        f"({count} mention{'s' if count != 1 else ''})."
    )

for i, insight in enumerate(business_insights, 1):
    print(f"{i}. {insight}")

1. Overall sentiment: 40.0% positive and 60.0% negative reviews.
2. Positive feedback was detected for battery (1 mention).
3. Positive feedback was detected for ease_of_use (1 mention).
4. Positive feedback was detected for quality (1 mention).
5. Positive feedback was detected for design (1 mention).
6. Positive feedback was detected for performance (1 mention).
7. Negative feedback was detected for delivery (1 mention).
8. Negative feedback was detected for packaging (1 mention).
9. Negative feedback was detected for size (1 mention).
10. Negative feedback was detected for fit (1 mention).
11. Negative feedback was detected for price (1 mention).


In [80]:
import json

business_insights_output = {
    "summary": {
        "total_reviews": len(final_pipeline_results),
        "positive_reviews": int(
            (final_pipeline_results["sentiment"] == "positive").sum()
        ),
        "negative_reviews": int(
            (final_pipeline_results["sentiment"] == "negative").sum()
        ),
        "positive_percentage": round(positive_pct, 2),
        "negative_percentage": round(negative_pct, 2)
    },
    "positive_aspects": {
        aspect: int(count)
        for aspect, count in positive_aspects.items()
    },
    "negative_aspects": {
        aspect: int(count)
        for aspect, count in negative_aspects.items()
    },
    "insights": business_insights
}

print(json.dumps(
    business_insights_output,
    indent=2,
    ensure_ascii=False
))

{
  "summary": {
    "total_reviews": 5,
    "positive_reviews": 2,
    "negative_reviews": 3,
    "positive_percentage": 40.0,
    "negative_percentage": 60.0
  },
  "positive_aspects": {
    "battery": 1,
    "ease_of_use": 1,
    "quality": 1,
    "design": 1,
    "performance": 1
  },
  "negative_aspects": {
    "delivery": 1,
    "packaging": 1,
    "size": 1,
    "fit": 1,
    "price": 1
  },
  "insights": [
    "Overall sentiment: 40.0% positive and 60.0% negative reviews.",
    "Positive feedback was detected for battery (1 mention).",
    "Positive feedback was detected for ease_of_use (1 mention).",
    "Positive feedback was detected for quality (1 mention).",
    "Positive feedback was detected for design (1 mention).",
    "Positive feedback was detected for performance (1 mention).",
    "Negative feedback was detected for delivery (1 mention).",
    "Negative feedback was detected for packaging (1 mention).",
    "Negative feedback was detected for size (1 mention).",
  

In [81]:
from pathlib import Path
import json

output_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "business_insights.json"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(
        business_insights_output,
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"Business insights saved to: {output_path}")

Business insights saved to: D:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\data\processed\business_insights.json


In [82]:
final_output_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "final_pipeline_results.csv"
)

final_pipeline_results.to_csv(
    final_output_path,
    index=False,
    encoding="utf-8-sig"
)

print(f"Final pipeline results saved to: {final_output_path}")

Final pipeline results saved to: D:\Portfolio\CustomerFeedbackIntelligence-(3NLPproject)\data\processed\final_pipeline_results.csv


In [83]:
print("Final output shape:", final_pipeline_results.shape)

print("\nFinal columns:")
for column in final_pipeline_results.columns:
    print("-", column)

display(final_pipeline_results.head())

Final output shape: (5, 10)

Final columns:
- review_id
- review
- sentiment
- sentiment_confidence
- emotion
- emotion_confidence
- topic_id
- topic
- aspects
- aspect_sentiments


,review_id,review,sentiment,sentiment_confidence,emotion,emotion_confidence,topic_id,topic,aspects,aspect_sentiments
0,1,The product is excellent and easy to use. The ...,positive,0.9586,joy,0.9991,10,10_easy_camera_install_use,"[battery, ease_of_use]","[{""aspect"": ""battery"", ""sentiment"": ""Positive""..."
1,2,The delivery was very slow and the packaging w...,negative,0.9909,sadness,0.8955,5,5_received_item_box_arrived,"[delivery, packaging]","[{""aspect"": ""delivery"", ""sentiment"": ""Negative..."
2,3,Great quality and beautiful design. I really l...,positive,0.9877,joy,0.9979,0,0_bag_great_quality_watch,"[quality, design]","[{""aspect"": ""quality"", ""sentiment"": ""Positive""..."
3,4,The size is too small and the fit is uncomfort...,negative,0.9945,fear,0.9990,2,2_size_comfortable_fit_shoes,"[size, fit]","[{""aspect"": ""size"", ""sentiment"": ""Negative"", ""..."
4,5,"The product works well, but the price is too h...",negative,0.8289,joy,0.9922,1,1_sound_work_charge_battery,"[price, performance]","[{""aspect"": ""price"", ""sentiment"": ""Negative"", ..."
